In [0]:
# Download hist`    orical HRRR forecasts for an explicit date range.
# Existing files are skipped, so rerunning resumes unfinished downloads.
# This is very similar to our scheduled downloader, but it is inteded to be run manually
# at the start of the archive creation and to fix an extended outage.

# I'm not putting high priority on backfilling this if some forecasts are missing due to problems
# in the course of regular downloading because that replicates the realistic data that
# will be available to make predictions

#
# Each gzip JSON file contains one weather location and one model run.
# fetched_at: when we actually downloaded the forecast, in UTC.
# forecast_initialized_at: the historical model initialization time, in UTC.
# location_id: weather cell identifier used by the station mapping.
# model, source, collection_type: forecast provenance.
# request_parameters: the API inputs used for this file.
# payload: the original response, including values, times, and units.

from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode
import gzip
import json
import time
import urllib3
from urllib3.util import Retry, Timeout

# Use job parameters when provided; otherwise use these interactive defaults.
parameters = {
    "run_mode": "production",
    "start_date": "2026-08-21",
    "end_date": "2026-09-26",
}
for parameter_name in parameters:
    try:
        parameters[parameter_name] = dbutils.widgets.get(parameter_name)
    except Exception:
        pass

RUN_MODE = parameters["run_mode"]
if RUN_MODE not in ("dev", "production"):
    raise ValueError("run_mode must be 'dev' or 'production'")

# Interpret both dates as midnight UTC, with the end excluded.
BACKFILL_START = datetime.strptime(
    parameters["start_date"], "%Y-%m-%d"
).replace(tzinfo=timezone.utc)

BACKFILL_END = datetime.strptime(
    parameters["end_date"], "%Y-%m-%d"
).replace(tzinfo=timezone.utc)

if BACKFILL_END <= BACKFILL_START:
    raise ValueError("end_date must be later than start_date")

if BACKFILL_END > datetime.now(timezone.utc):
    raise ValueError("Use an end_date no later than today in UTC")

# Use the same configuration and archive paths as the scheduled downloader.
WEATHER_ROOT = Path("/Volumes/citibike_project/citibike/raw/weather_forecast")
RUN_DIRECTORY = (
    WEATHER_ROOT if RUN_MODE == "production" else WEATHER_ROOT / "_dev"
)
ARCHIVE_ROOT = RUN_DIRECTORY / "hrrr"
API_URL = "https://single-runs-api.open-meteo.com/v1/forecast"

WEATHER_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "snowfall",
    "snow_depth",
    "wind_speed_10m",
    "cloud_cover",
]

configuration = json.loads(
    (RUN_DIRECTORY / "reference/hrrr_locations.json").read_text()
)
if not configuration["locations"]:
    raise ValueError("The weather location list is empty")

# Build the list of 00, 06, 12, and 18 UTC runs within the requested dates.
number_of_runs = int(
    (BACKFILL_END - BACKFILL_START).total_seconds() / (6 * 3600)
)
run_times = [
    BACKFILL_START + timedelta(hours=6 * offset)
    for offset in range(number_of_runs)
]

# Retry temporary failures, respecting the API's Retry-After instructions.
http = urllib3.PoolManager(
    timeout=Timeout(connect=10, read=60),
    retries=Retry(
        total=3,
        backoff_factor=5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods={"GET"},
        respect_retry_after_header=True,
    ),
)

print(f"RUN_MODE = {RUN_MODE}")
print(f"Archive: {ARCHIVE_ROOT}")
print(f"Start: {BACKFILL_START.isoformat()}")
print(f"End, exclusive: {BACKFILL_END.isoformat()}")
print(f"Expected files: {len(run_times) * len(configuration['locations']):,}")

total_downloaded = total_skipped = 0

for run_time in run_times:
    downloaded = skipped = 0

    # Each extended run should contain initialization plus hours 1 through 48.
    expected_times = [
        (run_time + timedelta(hours=hour)).strftime("%Y-%m-%dT%H:%M")
        for hour in range(49)
    ]

    for location in configuration["locations"]:

        # Preserve existing files and resume only the missing downloads.
        output_path = (
            ARCHIVE_ROOT / location["location_id"]
            / run_time.strftime("year=%Y/month=%m/day=%d")
            / f"hrrr_{run_time:%Y%m%dT%H%M%SZ}.json.gz"
        )
        if output_path.exists():
            skipped += 1
            continue

        # Request this specific historical run using the saved grid settings.
        request_parameters = {
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "models": configuration["model"],
            **configuration["grid_options"],
            "run": run_time.strftime("%Y-%m-%dT%H:%M"),
            "hourly": ",".join(WEATHER_VARIABLES),
            "forecast_hours": 49,
            "timezone": "UTC",
            "temperature_unit": "celsius",
            "wind_speed_unit": "ms",
            "precipitation_unit": "mm",
        }

        response = http.request(
            "GET", f"{API_URL}?{urlencode(request_parameters)}"
        )

        # A missing historical run needs attention rather than a silent skip.
        if response.status != 200:
            raise RuntimeError(
                f"{run_time.isoformat()}, {location['location_id']}: "
                f"HTTP {response.status}: {response.data.decode()}"
            )

        payload = json.loads(response.data)
        fetched_at = datetime.now(timezone.utc).isoformat()

        # Confirm that the selected weather cell matches our configuration.
        returned_id = (
            f"hrrr_{payload['latitude']:.6f}_{payload['longitude']:.6f}"
        )
        if returned_id != location["location_id"]:
            raise ValueError(
                f"Grid cell changed for {location['location_id']}: {returned_id}"
            )

        # Verify forecast completeness before saving the response.
        hourly = payload["hourly"]
        if hourly["time"] != expected_times:
            raise ValueError(f"Unexpected forecast times for {run_time}")

        for variable in WEATHER_VARIABLES:
            values = hourly[variable]
            required_values = (
                values[1:]
                if variable in ("precipitation", "snowfall")
                else values
            )
            if len(values) != 49 or any(
                value is None for value in required_values
            ):
                raise ValueError(
                    f"Incomplete {variable}: "
                    f"{location['location_id']}, {run_time}"
                )

        # Record the actual download time and identify this as historical backfill.
        record = {
            "fetched_at": fetched_at,
            "forecast_initialized_at": run_time.isoformat(),
            "location_id": location["location_id"],
            "model": configuration["model"],
            "source": "open_meteo_single_runs",
            "collection_type": "historical_backfill",
            "request_parameters": request_parameters,
            "payload": payload,
        }

        # Complete the compressed file before giving it its final archive name.
        output_path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = output_path.with_suffix(".gz.tmp")

        with gzip.open(temporary_path, "wt", encoding="utf-8") as archive:
            json.dump(record, archive)

        temporary_path.replace(output_path)
        downloaded += 1
        time.sleep(1)

    total_downloaded += downloaded
    total_skipped += skipped

    print(
        f"{run_time.isoformat()}: "
        f"downloaded {downloaded}, already saved {skipped}"
    )

http.clear()

print(f"Finished: {total_downloaded:,} downloaded; {total_skipped:,} already saved.")